In [1]:
import random
import cobra
import escher
import pandas as pd
import numpy as np
import json
import csv
from cobra.io import save_json_model
from cobra import Reaction
from collections import Counter
from cobra.flux_analysis.loopless import add_loopless, loopless_solution

In [2]:
cobra.Configuration().solver = "gurobi"

In [3]:
# model_path = './models/iMT1026v3.xml'
model_path = './models/iMT1026v3jup.xml'

model = cobra.io.read_sbml_model(model_path)

model

CobraSBMLError: Something went wrong reading the SBML model. Most likely the SBML model is not valid. Please check that your model is valid using the `cobra.io.sbml.validate_sbml_model` function or via the online validator at https://sbml.org/validator_servlet/ .
	`(model, errors) = validate_sbml_model(filename)`
If the model is valid and cannot be read please open an issue at https://github.com/opencobra/cobrapy/issues .

In [ ]:
model.summary()

In [ ]:
# Change grwoth on glycerol to growth on 60% glucose and 40% methanol
# Add reaction describing this growth

biomass_gly = model.reactions.get_by_id('BIOMASS_glyc')
biomass_gly.bounds = (0,0)
biomass_gly

carbs = model.metabolites.get_by_id('CARBOHYDRATES_c')
DNA = model.metabolites.get_by_id('DNA_c')
lipids = model.metabolites.get_by_id('LIPIDS_c')
prot = model.metabolites.get_by_id('PROTEIN_c')
RNA = model.metabolites.get_by_id('RNA_c')
atp = model.metabolites.get_by_id('atp_c')
cof = model.metabolites.get_by_id('cof_c')
h2o = model.metabolites.get_by_id('h2o_c')
adp = model.metabolites.get_by_id('adp_c')
biomass = model.metabolites.get_by_id('biomass_c')
h = model.metabolites.get_by_id('h_c')
pi = model.metabolites.get_by_id('pi_c')

biomass_glc_meoh = Reaction ('biomass_glc_meoh_60/40')
biomass_glc_meoh.name = 'Biomass composition (g/g) - 60/40 Glucose/Methanol'
biomass_glc_meoh.add_metabolites({carbs: -0.33,
                                   DNA: -0.001,
                                   lipids: -0.042,
                                   prot: -0.49,
                                   RNA: -0.058,
                                   atp: -63.85,
                                   cof: -1,
                                   h2o: -63.85,
                                   adp: 63.85,
                                   pi: 63.85,
                                   biomass: 1,
                                   h: 63.85})
biomass_glc_meoh

In [ ]:
# Add new biomass reaction to the model

print (len(model.reactions))
model.add_reactions([biomass_glc_meoh])
print (len(model.reactions))

In [ ]:
#Tanquem l'entrada de Glicerol
glycerol_exchange = model.exchanges.get_by_id('Ex_glyc')
glycerol_exchange.bounds = (0,0)
glycerol_exchange

In [ ]:
# Add qmeoh constraints observed in chemostat cultivations
methanol_exchange = model.exchanges.get_by_id('Ex_meoh')
methanol_exchange.bounds = (-1.24, -1.18)
methanol_exchange

In [ ]:
# Add qgluc constraints observed in chemostat cultivations
glucose_exchange = model.reactions.get_by_id('Ex_glc_D')
glucose_exchange.bounds = (-0.72, -0.7)
glucose_exchange

In [ ]:
# Change the reactions for the synthesis of lipids, proteins and sterols from glycerol to those from glucose

model.reactions.get_by_id('LIPIDS_glyc').bounds = (0,0)
model.reactions.get_by_id('PROTEINS_glyc').bounds = (0,0)
model.reactions.get_by_id('STEROLS_glyc').bounds = (0,0)

# note: these reactions from glucose do not have the glucose specification
model.reactions.get_by_id('LIPIDS').bounds = (0,1000)
model.reactions.get_by_id('PROTEINS').bounds = (0,1000)
model.reactions.get_by_id('STEROLS').bounds = (0,1000)

In [ ]:
# ATP maintenance requirement (NGAME = Non-Growth Associated Maintenance Energy) based on previous studies on 
# the growth of X33-ROL on Gluc/MeOH from the group (Eric's Master Thesis)

model.reactions.get_by_id('ATPM').bounds = (1.96, 1000)

In [ ]:
rolAA = model.reactions.get_by_id('rolAA')
rolAA.bounds = (0,0)
rolRNA = model.reactions.get_by_id('rolRNA')
rolRNA.bounds = (0,0)
rolDNA = model.reactions.get_by_id('rolDNA')
rolDNA.bounds = (0,0)
pROL = model.reactions.get_by_id('pROL')
pROL.bounds = (0,0)
Rol_transport =  model.reactions.get_by_id('ROLt')
Rol_transport.bounds = (0,0)
ROL_exchange = model.exchanges.get_by_id('Ex_rol')
#ROL_exchange.bounds = (0.0003773,1000)
ROL_exchange.bounds = (0,0) # Inferit de Bradford

pFAB = model.reactions.get_by_id('pFAB')
pFAB.bounds = (0,0)
fabAA = model.reactions.get_by_id('fabAA')
fabAA.bounds = (0,0)
fabt = model.reactions.get_by_id('fabt')
fabt.bounds = (0,0)
fabRNA = model.reactions.get_by_id('fabRNA')
fabRNA.bounds = (0,0)
fabDNA = model.reactions.get_by_id('fabDNA')
FAB_exchange = model.exchanges.get_by_id('Ex_fab')
FAB_exchange.bounds = (0,0)


# Extra reactions that must be closed for simulations to run smoothly:

APAT2r = model.reactions.get_by_id('APAT2r')
APAT2r.bounds = (0,0) # reaction not present in Pichia, it is yet to be removed from the model

MMSAD3 = model.reactions.get_by_id('MMSAD3')
MMSAD3.bounds = (0,0) # The reduction reaction of MSA into Acetil-CoA it is due to an unspecific effect. Reaction
# under evaluation of being kept or not.

In [ ]:
# REACTION RATIOS PER GLUCOSA METANOL

ReactionRatio1 = model.problem.Constraint(model.reactions.CSm.flux_expression - model.reactions.ACONTm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio1)

ReactionRatio2 = model.problem.Constraint(model.reactions.AKGDam.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio2)

ReactionRatio8 = model.problem.Constraint(68*model.reactions.AKGDam.flux_expression - 46*model.reactions.CSm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio8)

ReactionRatio9 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression - model.reactions.GND.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio9)

ReactionRatio10 = model.problem.Constraint(0.8*model.reactions.MDHm.flux_expression - model.reactions.FUMm.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio10)

ReactionRatio12 = model.problem.Constraint(35*model.reactions.PYK.flux_expression - 143*model.reactions.PC.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio12)

ReactionRatio13 = model.problem.Constraint(68*model.reactions.PYK.flux_expression - 143*model.reactions.PYRt2m.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio13)


ReactionRatio14 = model.problem.Constraint(40*model.reactions.GAPD.flux_expression - 145*model.reactions.FBA.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio14)


ReactionRatio16 = model.problem.Constraint(model.reactions.GAPD.flux_expression - model.reactions.PYK.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio16)

ReactionRatio20 = model.problem.Constraint(0.18*model.reactions.FALDtx.flux_expression - 0.82*model.reactions.DAS.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio20)

ReactionRatio21 = model.problem.Constraint(model.reactions.G6PDH2.flux_expression + 0.4*model.reactions.Ex_glc_D.flux_expression,
                                         lb=0,
                                         ub=0)
model.add_cons_vars(ReactionRatio21)

In [ ]:
pfba_WT = cobra.flux_analysis.pfba(model)

In [ ]:
pfba_WT.fluxes['Ex_biomass']

In [ ]:
nadph_producing_reactions = []

nadph_c = model.metabolites.get_by_id("nadph_c")

for reaction in model.reactions:
    coefficient = reaction.metabolites.get(nadph_c, 0)

    if coefficient > 0:
        nadph_producing_reactions.append({
            "ID": reaction.id,
            "Name": reaction.name,
            "Reaction": reaction.reaction,
            "NADPH stoichiometry": coefficient
        })

for r in nadph_producing_reactions:
    print(r["ID"], "|", r["Name"], "|", r["NADPH stoichiometry"])

In [ ]:
increase = 0.20  # 20% increase

nadph_c = model.metabolites.get_by_id("nadph_c")

with model:

    # --------------------------------------------------
    # 1. Identify NADPH-producing reactions
    #    based on the pFBA_WT flux direction
    # --------------------------------------------------

    nadph_producing_reactions = []

    for reaction in model.reactions:

        nadph_stoich = reaction.metabolites.get(nadph_c, 0)
        reference_flux = pfba_WT.fluxes[reaction.id]

        # Net NADPH production in the WT pFBA solution
        if reference_flux * nadph_stoich > 0:

            target_flux = reference_flux * (1 + increase)

            nadph_producing_reactions.append({
                "ID": reaction.id,
                "Name": reaction.name,
                "Reference flux": reference_flux,
                "NADPH stoichiometry": nadph_stoich,
                "Reference NADPH production": (
                    reference_flux * nadph_stoich
                ),
                "Target flux": target_flux
            })

            # Require at least the increased flux
            reaction.lower_bound = target_flux


    # --------------------------------------------------
    # 2. Perform MOMA using pfba_WT as reference
    # --------------------------------------------------

    moma_result = cobra.flux_analysis.moma(
        model,
        solution=pfba_WT,
        linear=False
    )

print("MOMA status:", moma_result.status)

In [ ]:
## Afegir aqui linea de codi per comprovar que la simulació MOMA té un turnover rate més alt que el pfba de ref

In [ ]:
# 2. Calculate flux changes

reaction_changes = []

for reaction in model.reactions:

    reaction_id = reaction.id

    reference_flux = pfba_WT.fluxes[reaction_id]
    moma_flux = moma_result.fluxes[reaction_id]

    absolute_change = abs(moma_flux - reference_flux)

    # Proportional change
    if abs(reference_flux) > 1e-12:
        proportional_change = absolute_change / abs(reference_flux)
    else:
        proportional_change = np.nan

    reaction_changes.append({
        "ID": reaction_id,
        "Name": reaction.name,
        "pFBA flux": reference_flux,
        "MOMA flux": moma_flux,
        "Absolute change": absolute_change,
        "Proportional change": proportional_change
    })

In [ ]:
flux_threshold = 1e-6

proportional_changes = sorted(
    [
        r for r in reaction_changes
        if abs(r["pFBA flux"]) > flux_threshold
    ],
    key=lambda x: x["Proportional change"],
    reverse=True
)

print("Reactions ordered by proportional flux change:\n")

for r in proportional_changes:
    print(
        r["ID"],
        "|", r["Name"],
        "| pFBA =", r["pFBA flux"],
        "| MOMA =", r["MOMA flux"],
        "| proportional change =", r["Proportional change"]
    )

In [ ]:
absolute_changes = sorted(
    reaction_changes,
    key=lambda x: x["Absolute change"],
    reverse=True
)

print("Reactions ordered by absolute flux change:\n")

for r in absolute_changes:
    print(
        r["ID"],
        "|", r["Name"],
        "| pFBA =", r["pFBA flux"],
        "| MOMA =", r["MOMA flux"],
        "| absolute change =", r["Absolute change"]
    )

# Version with Randomness

In [ ]:
increase_min = 0.05   # 5%
increase_max = 0.10   # 10%

flux_threshold = 1e-6

# Increase NADPH-producing reactions + MOMA

nadph_c = model.metabolites.get_by_id("nadph_c")

with model:

    # Identify NADPH-producing reactions
    nadph_producing_reactions = []

    for reaction in model.reactions:

        nadph_stoich = reaction.metabolites.get(nadph_c, 0)
        reference_flux = pfba_WT.fluxes[reaction.id]

        # Only reactions that actually produce NADPH
        # in the pFBA_WT solution
        if reference_flux * nadph_stoich > 0:

            # Random increase between 5 and 20%
            increase = np.random.uniform(
                increase_min,
                increase_max
            )

            target_flux = reference_flux * (1 + increase)

            nadph_producing_reactions.append({
                "ID": reaction.id,
                "Name": reaction.name,
                "Reference flux": reference_flux,
                "NADPH stoichiometry": nadph_stoich,
                "Increase": increase,
                "Target flux": target_flux
            })

            # Set new lower bound
            reaction.lower_bound = target_flux

    moma_result = cobra.flux_analysis.moma(
        model,
        solution=pfba_WT,
        linear=True
    )
    
# Calculate flux changes

reaction_changes = []

for reaction in model.reactions:

    reaction_id = reaction.id

    reference_flux = pfba_WT.fluxes[reaction_id]
    moma_flux = moma_result.fluxes[reaction_id]

    absolute_change = abs(
        moma_flux - reference_flux
    )

    # Avoid division by zero 
    if abs(reference_flux) > flux_threshold:

        proportional_change = (
            absolute_change / abs(reference_flux)
        )

    else:

        proportional_change = np.nan

    reaction_changes.append({
        "ID": reaction_id,
        "Name": reaction.name,
        "pFBA flux": reference_flux,
        "MOMA flux": moma_flux,
        "Absolute change": absolute_change,
        "Proportional change": proportional_change
    })

# Top 10 proportional changes


proportional_changes = sorted(
    [
        r for r in reaction_changes
        if abs(r["pFBA flux"]) > flux_threshold
    ],
    key=lambda x: x["Proportional change"],
    reverse=True
)

print("\n===== TOP 10 PROPORTIONAL CHANGES =====\n")

for r in proportional_changes[:10]:

    print(
        r["ID"],
        "|", r["Name"],
        "| pFBA =", round(r["pFBA flux"], 6),
        "| MOMA =", round(r["MOMA flux"], 6),
        "| change =",
        round(r["Proportional change"] * 100, 2),
        "%"
    )

# Top 10 absolute changes

absolute_changes = sorted(
    reaction_changes,
    key=lambda x: x["Absolute change"],
    reverse=True
)

print("\n===== TOP 10 ABSOLUTE CHANGES =====\n")

for r in absolute_changes[:10]:

    print(
        r["ID"],
        "|", r["Name"],
        "| pFBA =", round(r["pFBA flux"], 6),
        "| MOMA =", round(r["MOMA flux"], 6),
        "| absolute change =",
        round(r["Absolute change"], 6)
    )

# Save reaction IDs to Excel

proportional_ids = [
    r["ID"] for r in proportional_changes[:10]
]

absolute_ids = [
    r["ID"] for r in absolute_changes[:10]
]

with pd.ExcelWriter(
    "results/RDXMAX_it.xlsx",
    engine="openpyxl"
) as writer:

    pd.DataFrame({
        "Reaction ID": proportional_ids
    }).to_excel(
        writer,
        sheet_name="Top 10 Proportional",
        index=False
    )

    pd.DataFrame({
        "Reaction ID": absolute_ids
    }).to_excel(
        writer,
        sheet_name="Top 10 Absolute",
        index=False
    )

print("\nResults saved to RDXMAX_it.xlsx")

In [ ]:
moma_result.fluxes['Ex_biomass']

In [ ]:
# NADPH turnover rate WT strain

positive_nadph_flux_sum_WT = 0.0
for reaction in nadph_c.reactions:
    flux = pfba_WT.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadph_c)
    # No totes les reaccions tenen 1 de coeficient estequiomètric pel cofactor
    #independentment si s'està produint o consumint
    positive_nadph_flux_sum_WT += abs(flux*coef)
    

print (positive_nadph_flux_sum_WT/2)

In [ ]:
# NADPH turnover rate mutant strain

positive_nadph_flux_sum_moma = 0.0
for reaction in nadph_c.reactions:
    flux = moma_result.fluxes[reaction.id]
    coef = reaction.get_coefficient(nadph_c)
    # No totes les reaccions tenen 1 de coeficient estequiomètric pel cofactor
    #independentment si s'està produint o consumint
    positive_nadph_flux_sum_moma += abs(flux*coef)
    

print (positive_nadph_flux_sum_moma/2)

In [ ]:
#Set the number of iterations to run

n_iterations = 20

proportional_results = []
absolute_results = []

for iteration in range(1, n_iterations + 1):

    # 1. Randomly increase each NADPH-producing reaction between a 5 and 10 %

    increase_min = 0.05
    increase_max = 0.10

    nadph_c = model.metabolites.get_by_id("nadph_c")

    with model:

        for reaction in model.reactions:

            nadph_stoich = reaction.metabolites.get(nadph_c, 0)
            reference_flux = pfba_WT.fluxes[reaction.id]

            if reference_flux * nadph_stoich > 0:

                increase = np.random.uniform(
                    increase_min,
                    increase_max
                )

                target_flux = reference_flux * (1 + increase)

                reaction.lower_bound = target_flux
       
        # 2. MOMA using pFBA_WT as reference
    
        moma_result = cobra.flux_analysis.moma(
            model,
            solution=pfba_WT,
            linear=True
        )

        # 3. Calculate changes

        reaction_changes = []

        for reaction in model.reactions:

            reaction_id = reaction.id

            reference_flux = pfba_WT.fluxes[reaction_id]
            moma_flux = moma_result.fluxes[reaction_id]

            absolute_change = abs(
                moma_flux - reference_flux
            )

            if abs(reference_flux) > 1e-6:

                proportional_change = (
                    absolute_change / abs(reference_flux)
                )

            else:

                proportional_change = np.nan

            reaction_changes.append({
                "ID": reaction_id,
                "Name": reaction.name,
                "pFBA flux": reference_flux,
                "MOMA flux": moma_flux,
                "Absolute change": absolute_change,
                "Proportional change": proportional_change
            })

        # 4. Top 10 proportional changes

        proportional_changes = sorted(
            [
                r for r in reaction_changes
                if not np.isnan(r["Proportional change"])
            ],
            key=lambda x: x["Proportional change"],
            reverse=True
        )

        # Use reaction IDs instead of names
        proportional_ids = [
            r["ID"] for r in proportional_changes[:10]
        ]

        # 5. Top 10 absolute changes

        absolute_changes = sorted(
            reaction_changes,
            key=lambda x: x["Absolute change"],
            reverse=True
        )

        # Use reaction IDs instead of names
        absolute_ids = [
            r["ID"] for r in absolute_changes[:10]
        ]

    # 6. Store results from this iteration

    proportional_results.append(
        [iteration] + proportional_ids
    )

    absolute_results.append(
        [iteration] + absolute_ids
    )

    print(f"Iteration {iteration}/{n_iterations} completed")

# 7. Create Excel file

columns = ["Iteration"] + [
    f"Top {i}" for i in range(1, 11)
]

df_proportional = pd.DataFrame(
    proportional_results,
    columns=columns
)

df_absolute = pd.DataFrame(
    absolute_results,
    columns=columns
)


with pd.ExcelWriter(
    "results/RDXMAX_results.xlsx",
    engine="openpyxl"
) as writer:

    df_proportional.to_excel(
        writer,
        sheet_name="Proportional changes",
        index=False
    )

    df_absolute.to_excel(
        writer,
        sheet_name="Absolute changes",
        index=False
    )

print("Excel file created: RDXMAX_results.xlsx")

### The next step is to count the number of times that reactions appear in the lists

In [ ]:
from collections import Counter

# Remove the Iteration column
proportional_ids = df_proportional.drop(columns=["Iteration"]).values.flatten()
absolute_ids = df_absolute.drop(columns=["Iteration"]).values.flatten()

# Count occurrences
proportional_counts = Counter(proportional_ids)
absolute_counts = Counter(absolute_ids)

# Convert to DataFrames and sort
df_proportional_counts = pd.DataFrame(
    proportional_counts.items(),
    columns=["Reaction ID", "Occurrences"]
).sort_values(
    "Occurrences",
    ascending=False
)

df_absolute_counts = pd.DataFrame(
    absolute_counts.items(),
    columns=["Reaction ID", "Occurrences"]
).sort_values(
    "Occurrences",
    ascending=False
)

print("\n===== PROPORTIONAL CHANGE FREQUENCY =====\n")
print(df_proportional_counts)

print("\n===== ABSOLUTE CHANGE FREQUENCY =====\n")
print(df_absolute_counts)

In [ ]:
# Transfer the results into an Excel file
with pd.ExcelWriter(
    "results/RDXMAX_frecuencies.xlsx",
    engine="openpyxl"
) as writer:

    df_proportional.to_excel(
        writer,
        sheet_name="Proportional changes",
        index=False
    )

    df_absolute.to_excel(
        writer,
        sheet_name="Absolute changes",
        index=False
    )

    df_proportional_counts.to_excel(
        writer,
        sheet_name="Proportional frequency",
        index=False
    )

    df_absolute_counts.to_excel(
        writer,
        sheet_name="Absolute frequency",
        index=False
    )